In [1]:
!pip install pandas requests pyarrow openpyxl -q

In [2]:
import requests
from pathlib import Path

# ------------------------------------------------------------
# IBTrACS North Indian Ocean CSV
# ------------------------------------------------------------
url = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.NI.list.v04r01.csv"

output_dir = Path("ibtracs_cyclone_data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

raw_file = output_dir / "ibtracs_NI_v04r01.csv"

response = requests.get(url, stream=True, timeout=300)
response.raise_for_status()

with open(raw_file, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print("Downloaded:", raw_file)
print("File size MB:", round(raw_file.stat().st_size / (1024 * 1024), 2))

Downloaded: ibtracs_cyclone_data/raw/ibtracs_NI_v04r01.csv
File size MB: 26.42


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_file = Path("ibtracs_cyclone_data/raw/ibtracs_NI_v04r01.csv")

# IBTrACS CSV often has a second row containing units.
# We read normally, then remove non-date/unit rows using ISO_TIME parsing.
df = pd.read_csv(raw_file, low_memory=False)

# Clean column names
df.columns = df.columns.str.strip().str.lower()

print("Original shape:", df.shape)
print("Columns:")
print(df.columns.tolist()[:50])

# Parse time
df["iso_time"] = pd.to_datetime(df["iso_time"], errors="coerce")

# Remove unit row / invalid rows
df = df.dropna(subset=["iso_time"])

# Convert key numeric columns
numeric_cols = [
    "lat", "lon", "wmo_wind", "wmo_pres", "usa_wind", "usa_pres",
    "usa_sshs", "dist2land", "landfall"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Cleaned shape:", df.shape)
print(df[["sid", "season", "name", "iso_time", "basin", "lat", "lon"]].head())

Original shape: (62482, 174)
Columns:
['sid', 'season', 'number', 'basin', 'subbasin', 'name', 'iso_time', 'nature', 'lat', 'lon', 'wmo_wind', 'wmo_pres', 'wmo_agency', 'track_type', 'dist2land', 'landfall', 'iflag', 'usa_agency', 'usa_atcf_id', 'usa_lat', 'usa_lon', 'usa_record', 'usa_status', 'usa_wind', 'usa_pres', 'usa_sshs', 'usa_r34_ne', 'usa_r34_se', 'usa_r34_sw', 'usa_r34_nw', 'usa_r50_ne', 'usa_r50_se', 'usa_r50_sw', 'usa_r50_nw', 'usa_r64_ne', 'usa_r64_se', 'usa_r64_sw', 'usa_r64_nw', 'usa_poci', 'usa_roci', 'usa_rmw', 'usa_eye', 'tokyo_lat', 'tokyo_lon', 'tokyo_grade', 'tokyo_wind', 'tokyo_pres', 'tokyo_r50_dir', 'tokyo_r50_long', 'tokyo_r50_short']


/tmp/ipykernel_2604/293571396.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["iso_time"] = pd.to_datetime(df["iso_time"], errors="coerce")


Cleaned shape: (62481, 174)
             sid season     name            iso_time basin   lat   lon
1  1842298N11080   1842  UNNAMED 1842-10-25 03:00:00    NI  10.9  80.3
2  1842298N11080   1842  UNNAMED 1842-10-25 06:00:00    NI  10.9  79.8
3  1842298N11080   1842  UNNAMED 1842-10-25 09:00:00    NI  10.8  79.4
4  1842298N11080   1842  UNNAMED 1842-10-25 12:00:00    NI  10.8  78.9
5  1842298N11080   1842  UNNAMED 1842-10-25 15:00:00    NI  10.8  78.4


In [4]:
# ------------------------------------------------------------
# Fixed project period
# ------------------------------------------------------------
start_date = "2020-01-01"
end_date = "2024-12-31 23:59:59"

df_period = df[
    (df["iso_time"] >= start_date) &
    (df["iso_time"] <= end_date)
].copy()

print("2020–2024 shape:", df_period.shape)
print("Storms:", df_period["sid"].nunique())
print(df_period[["sid", "season", "name", "iso_time", "lat", "lon", "wmo_wind", "wmo_pres"]].head())

2020–2024 shape: (1916, 174)
Storms: 58
                 sid season    name            iso_time  lat   lon  wmo_wind  \
60435  2020136N10088   2020  AMPHAN 2020-05-15 06:00:00  9.5  87.5       NaN   
60436  2020136N10088   2020  AMPHAN 2020-05-15 09:00:00  9.5  87.2       NaN   
60437  2020136N10088   2020  AMPHAN 2020-05-15 12:00:00  9.5  87.0       NaN   
60438  2020136N10088   2020  AMPHAN 2020-05-15 15:00:00  9.5  86.8       NaN   
60439  2020136N10088   2020  AMPHAN 2020-05-15 18:00:00  9.5  86.6       NaN   

       wmo_pres  
60435       NaN  
60436       NaN  
60437       NaN  
60438       NaN  
60439       NaN  


In [5]:
def assign_study_basin(lat, lon):
    # Bay of Bengal: 5°N–22°N, 80°E–98°E
    if 5 <= lat <= 22 and 80 <= lon <= 98:
        return "Bay_of_Bengal"

    # Arabian Sea: 5°N–22°N, 60°E–75°E
    if 5 <= lat <= 22 and 60 <= lon <= 75:
        return "Arabian_Sea"

    return "Outside_Study_Box"

df_period["study_basin"] = df_period.apply(
    lambda row: assign_study_basin(row["lat"], row["lon"]),
    axis=1
)

print(df_period["study_basin"].value_counts())

study_basin
Bay_of_Bengal        982
Outside_Study_Box    639
Arabian_Sea          295
Name: count, dtype: int64


In [6]:
def cyclone_category_from_wind_kts(wind):
    if pd.isna(wind):
        return "Unknown"
    elif wind < 34:
        return "Depression"
    elif wind < 64:
        return "Tropical_Storm"
    elif wind < 83:
        return "Category_1"
    elif wind < 96:
        return "Category_2"
    elif wind < 113:
        return "Category_3"
    elif wind < 137:
        return "Category_4"
    else:
        return "Category_5"

# Prefer WMO wind. If missing, use USA wind.
df_period["wind_speed_kts"] = df_period["wmo_wind"]

if "usa_wind" in df_period.columns:
    df_period["wind_speed_kts"] = df_period["wind_speed_kts"].fillna(df_period["usa_wind"])

df_period["storm_category"] = df_period["wind_speed_kts"].apply(cyclone_category_from_wind_kts)

# Prefer WMO pressure. If missing, use USA pressure.
df_period["central_pressure_mb"] = df_period["wmo_pres"]

if "usa_pres" in df_period.columns:
    df_period["central_pressure_mb"] = df_period["central_pressure_mb"].fillna(df_period["usa_pres"])

print(df_period[["sid", "name", "iso_time", "study_basin", "wind_speed_kts", "central_pressure_mb", "storm_category"]].head())

                 sid    name            iso_time    study_basin  \
60435  2020136N10088  AMPHAN 2020-05-15 06:00:00  Bay_of_Bengal   
60436  2020136N10088  AMPHAN 2020-05-15 09:00:00  Bay_of_Bengal   
60437  2020136N10088  AMPHAN 2020-05-15 12:00:00  Bay_of_Bengal   
60438  2020136N10088  AMPHAN 2020-05-15 15:00:00  Bay_of_Bengal   
60439  2020136N10088  AMPHAN 2020-05-15 18:00:00  Bay_of_Bengal   

       wind_speed_kts  central_pressure_mb storm_category  
60435            20.0               1007.0     Depression  
60436            25.0               1005.0     Depression  
60437            30.0               1002.0     Depression  
60438            30.0               1002.0     Depression  
60439            30.0               1001.0     Depression  


In [7]:
required_columns = [
    "sid",
    "season",
    "name",
    "iso_time",
    "basin",
    "subbasin",
    "study_basin",
    "lat",
    "lon",
    "nature",
    "wind_speed_kts",
    "central_pressure_mb",
    "storm_category"
]

# Add optional columns if available
optional_columns = [
    "wmo_agency",
    "usa_agency",
    "usa_status",
    "usa_sshs",
    "dist2land",
    "landfall",
    "track_type"
]

available_columns = [c for c in required_columns + optional_columns if c in df_period.columns]

ibtracs_clean = df_period[available_columns].copy()

ibtracs_clean = ibtracs_clean.rename(columns={
    "sid": "cyclone_id",
    "name": "storm_name",
    "iso_time": "timestamp",
    "lat": "cyclone_center_lat",
    "lon": "cyclone_center_lon",
    "basin": "ibtracs_basin",
    "subbasin": "ibtracs_subbasin"
})

print("Final clean shape:", ibtracs_clean.shape)
ibtracs_clean.head()

Final clean shape: (1916, 20)


,cyclone_id,season,storm_name,timestamp,ibtracs_basin,ibtracs_subbasin,study_basin,cyclone_center_lat,cyclone_center_lon,nature,wind_speed_kts,central_pressure_mb,storm_category,wmo_agency,usa_agency,usa_status,usa_sshs,dist2land,landfall,track_type
60435,2020136N10088,2020,AMPHAN,2020-05-15 06:00:00,NI,BB,Bay_of_Bengal,9.5,87.5,DS,20.0,1007.0,Depression,,jtwc_io,DB,-3,662,641.0,main
60436,2020136N10088,2020,AMPHAN,2020-05-15 09:00:00,NI,BB,Bay_of_Bengal,9.5,87.2,DS,25.0,1005.0,Depression,,,DB,-3,631,610.0,main
60437,2020136N10088,2020,AMPHAN,2020-05-15 12:00:00,NI,BB,Bay_of_Bengal,9.5,87.0,TS,30.0,1002.0,Depression,,jtwc_io,TD,-1,610,589.0,main
60438,2020136N10088,2020,AMPHAN,2020-05-15 15:00:00,NI,BB,Bay_of_Bengal,9.5,86.8,TS,30.0,1002.0,Depression,,,TD,-1,589,569.0,main
60439,2020136N10088,2020,AMPHAN,2020-05-15 18:00:00,NI,BB,Bay_of_Bengal,9.5,86.6,TS,30.0,1001.0,Depression,,jtwc_io,TD,-1,569,559.0,main


In [8]:
from pathlib import Path

processed_dir = Path("ibtracs_cyclone_data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

csv_file = processed_dir / "ibtracs_NI_cyclones_2020_2024_clean.csv"
excel_file = processed_dir / "ibtracs_NI_cyclones_2020_2024_clean.xlsx"
parquet_file = processed_dir / "ibtracs_NI_cyclones_2020_2024_clean.parquet"

ibtracs_clean.to_csv(csv_file, index=False)
ibtracs_clean.to_excel(excel_file, index=False)
ibtracs_clean.to_parquet(parquet_file, index=False)

print("Saved CSV:", csv_file)
print("Saved Excel:", excel_file)
print("Saved Parquet:", parquet_file)

Saved CSV: ibtracs_cyclone_data/processed/ibtracs_NI_cyclones_2020_2024_clean.csv
Saved Excel: ibtracs_cyclone_data/processed/ibtracs_NI_cyclones_2020_2024_clean.xlsx
Saved Parquet: ibtracs_cyclone_data/processed/ibtracs_NI_cyclones_2020_2024_clean.parquet


In [9]:
ibtracs_study_area = ibtracs_clean[
    ibtracs_clean["study_basin"].isin(["Bay_of_Bengal", "Arabian_Sea"])
].copy()

study_csv = processed_dir / "ibtracs_cyclone_points_bob_as_2020_2024.csv"
study_excel = processed_dir / "ibtracs_cyclone_points_bob_as_2020_2024.xlsx"
study_parquet = processed_dir / "ibtracs_cyclone_points_bob_as_2020_2024.parquet"

ibtracs_study_area.to_csv(study_csv, index=False)
ibtracs_study_area.to_excel(study_excel, index=False)
ibtracs_study_area.to_parquet(study_parquet, index=False)

print("Study-area cyclone points:", ibtracs_study_area.shape)
print("Saved:", study_excel)

Study-area cyclone points: (1277, 20)
Saved: ibtracs_cyclone_data/processed/ibtracs_cyclone_points_bob_as_2020_2024.xlsx


In [10]:
event_summary = (
    ibtracs_clean.groupby(["cyclone_id", "storm_name", "season"], as_index=False)
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        max_wind_kts=("wind_speed_kts", "max"),
        min_pressure_mb=("central_pressure_mb", "min"),
        first_lat=("cyclone_center_lat", "first"),
        first_lon=("cyclone_center_lon", "first"),
        last_lat=("cyclone_center_lat", "last"),
        last_lon=("cyclone_center_lon", "last"),
        affected_regions=("study_basin", lambda x: ",".join(sorted(set(x.dropna())))),
        track_points=("timestamp", "count")
    )
)

event_summary["max_category"] = event_summary["max_wind_kts"].apply(cyclone_category_from_wind_kts)

event_csv = processed_dir / "ibtracs_cyclone_event_summary_2020_2024.csv"
event_excel = processed_dir / "ibtracs_cyclone_event_summary_2020_2024.xlsx"
event_parquet = processed_dir / "ibtracs_cyclone_event_summary_2020_2024.parquet"

event_summary.to_csv(event_csv, index=False)
event_summary.to_excel(event_excel, index=False)
event_summary.to_parquet(event_parquet, index=False)

print("Event summary shape:", event_summary.shape)
event_summary.head()

Event summary shape: (58, 14)


,cyclone_id,storm_name,season,start_time,end_time,max_wind_kts,min_pressure_mb,first_lat,first_lon,last_lat,last_lon,affected_regions,track_points,max_category
0,2020136N10088,AMPHAN,2020,2020-05-15 06:00:00,2020-05-21 12:00:00,130.0,920.0,9.5,87.5,25.4,89.6,"Bay_of_Bengal,Outside_Study_Box",51,Category_4
1,2020150N17054,UNNAMED,2020,2020-05-29 09:00:00,2020-05-31 18:00:00,25.0,1000.0,17.3,54.3,16.8,52.9,Outside_Study_Box,20,Depression
2,2020153N13072,NISARGA,2020,2020-05-31 18:00:00,2020-06-04 06:00:00,60.0,984.0,12.9,72.2,22.2,77.6,"Arabian_Sea,Outside_Study_Box",29,Tropical_Storm
3,2020285N15087,UNNAMED,2020,2020-10-11 00:00:00,2020-10-14 06:00:00,30.0,996.0,15.3,86.5,17.7,77.0,"Bay_of_Bengal,Outside_Study_Box",27,Depression
4,2020291N18069,UNNAMED,2020,2020-10-17 03:00:00,2020-10-18 18:00:00,25.0,1003.0,17.8,69.0,17.8,64.3,Arabian_Sea,14,Depression


In [11]:
import shutil
from google.colab import files

zip_name = "ibtracs_cyclone_data_2020_2024"

shutil.make_archive(zip_name, "zip", "ibtracs_cyclone_data")

files.download(zip_name + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>